# Transfer Learning e Fine-Tuning
## Riutilizzare un modello già allenato invece di partire da zero

Nelle lezioni precedenti abbiamo costruito e allenato i modelli **da zero**: pesi casuali, e la rete impara tutto dai nostri dati.

Oggi usiamo un approccio diverso, ed è quello che si usa più spesso nel lavoro reale. Prendiamo una rete **già allenata da Google su milioni di immagini** e la adattiamo al nostro problema.

**Obiettivi della lezione**
1. Capire *perché* un modello allenato su un altro problema può aiutarci.
2. Conoscere le due strategie principali: **feature extraction** e **fine-tuning**.
3. Misurare i vantaggi con esperimenti veri, confrontando accuratezza, tempo e quantità di dati necessari.


**Gli esperimenti di oggi** (stesso dataset, stesse condizioni):

| | Esperimento | Cosa alleniamo |
|---|---|---|
| A | CNN da zero | tutta la rete, partendo da pesi casuali |
| B | Feature extraction | solo l'ultimo strato (la rete pre-allenata è "congelata") |
| C | Fine-tuning | l'ultimo strato più gli ultimi blocchi della rete pre-allenata |
| D | Quanti dati servono? | A e B con il 10% e il 30% dei dati |

---
## 0. ⚙️ Configurare la TPU (da fare PRIMA di eseguire qualsiasi cella)

Con l'account gratuito di Colab useremo una **TPU** (Tensor Processing Unit), un acceleratore progettato da Google proprio per le reti neurali.

1. Menu **Runtime → Cambia tipo di runtime**
2. In *Acceleratore hardware* scegli **TPU** (una qualsiasi di quelle proposte)
3. **Salva** e poi esegui le celle dall'alto verso il basso

> **Perché Keras + JAX?** Sulla TPU di Colab la libreria più semplice da usare è **JAX** (anch'essa di Google). Keras 3 ci permette di scrivere il codice in modo semplice e di usare JAX "sotto il cofano". I concetti sono identici a quelli visti con PyTorch: layer, loss, ottimizzatore, epoche.

> ⚠️ Se la TPU non è disponibile, il notebook funziona comunque su CPU in **modalità veloce**, con meno immagini e meno epoche. I risultati saranno meno netti, ma l'idea si vede lo stesso.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"   # deve stare PRIMA di "import keras"

import time, tarfile, urllib.request, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import jax
import keras
from keras import layers

PIATTAFORMA = jax.devices()[0].platform      # 'tpu', 'gpu' oppure 'cpu'
print("Keras", keras.__version__, "| backend:", keras.backend.backend())
print("Dispositivi trovati:", jax.devices())

if PIATTAFORMA == "tpu":
    # La TPU lavora in modo nativo con il formato bfloat16: calcoli più veloci, metà memoria
    keras.mixed_precision.set_global_policy("mixed_bfloat16")
    print("\n✅ TPU attiva: tutto pronto!")
elif PIATTAFORMA == "gpu":
    print("\n✅ GPU attiva: tutto pronto!")
else:
    print("\n⚠️ Nessun acceleratore: userò la MODALITÀ VELOCE (meno dati, meno epoche).")
    print("   Controlla: Runtime → Cambia tipo di runtime → TPU")

MODALITA_VELOCE = (PIATTAFORMA == "cpu")
keras.utils.set_random_seed(42)

---
## 1. 🧠 L'idea: perché un modello "già allenato" ci aiuta?

Una CNN allenata su **ImageNet** (circa 1,4 milioni di immagini e 1000 categorie: animali, oggetti, veicoli, cibo...) impara una **gerarchia di caratteristiche**:

| Strati | Cosa riconoscono | Utili per i nostri fiori? |
|---|---|---|
| Primi strati | bordi, linee, colori | ✅ sì, sono universali |
| Strati intermedi | texture, forme, pattern (petali, venature, cerchi) | ✅ quasi sempre |
| Ultimi strati | concetti specifici di ImageNet ("golden retriever", "tazza") | ❌ no, vanno sostituiti |

**Analogia:** un pittore esperto che impara a dipingere fiori non riparte da "come si tiene un pennello". Riusa tutto quello che sa (colori, prospettiva, luce) e impara solo la parte nuova.

### Le due strategie

**1) Feature extraction** (estrazione di caratteristiche)
- Teniamo la rete pre-allenata **congelata**, cioè i suoi pesi non cambiano.
- Togliamo il suo classificatore finale e ne mettiamo uno **nuovo**, adatto alle nostre 5 classi.
- Alleniamo **solo** quel nuovo strato: pochi parametri, veloce, difficile andare in overfitting.

**2) Fine-tuning** (rifinitura)
- Partiamo dal modello del punto 1.
- **Scongeliamo** gli ultimi blocchi della rete pre-allenata e li alleniamo insieme al classificatore.
- Usiamo un **learning rate molto basso**: vogliamo *ritoccare* i pesi, non distruggerli.

### Il costo, in numeri
- Allenare un modello come MobileNetV2 su ImageNet richiede **giorni** di calcolo su hardware dedicato.
- Noi useremo quel lavoro gratis e lo adatteremo in **pochi minuti**, con circa 2500 immagini.

---
## 2. 🌷 Il dataset: TF Flowers

Circa **3670 foto di fiori** in **5 classi**: margherita, tarassaco, rosa, girasole, tulipano.

È un dataset piccolo, e proprio per questo realistico: in azienda raramente si hanno milioni di immagini etichettate.

In [ ]:
URL = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
CARTELLA = pathlib.Path("dati")
CARTELLA_FOTO = CARTELLA / "flower_photos"
CARTELLA.mkdir(exist_ok=True)

if not CARTELLA_FOTO.exists():
    print("Scarico il dataset (circa 220 MB)...")
    archivio = CARTELLA / "flower_photos.tgz"
    urllib.request.urlretrieve(URL, archivio)
    with tarfile.open(archivio) as tar:
        tar.extractall(CARTELLA, filter="data")
    archivio.unlink()
print("✅ Dataset pronto in", CARTELLA_FOTO)

In [ ]:
# --- Parametri principali ---
IMG = 128 if MODALITA_VELOCE else 160          # lato delle immagini in pixel
BATCH = 32 if MODALITA_VELOCE else 64
MAX_PER_CLASSE = 150 if MODALITA_VELOCE else None   # in modalità veloce usiamo meno foto

NOMI_IT = {"daisy": "margherita", "dandelion": "tarassaco", "roses": "rosa",
           "sunflowers": "girasole", "tulips": "tulipano"}
CLASSI_EN = sorted(d.name for d in CARTELLA_FOTO.iterdir() if d.is_dir())
CLASSI = [NOMI_IT.get(c, c) for c in CLASSI_EN]
N_CLASSI = len(CLASSI)

# Carichiamo tutte le immagini in un array NumPy (le teniamo in uint8: occupano 4 volte meno memoria)
t0 = time.time()
X, y = [], []
for indice, nome in enumerate(CLASSI_EN):
    file = sorted((CARTELLA_FOTO / nome).glob("*.jpg"))[:MAX_PER_CLASSE]
    for f in file:
        img = Image.open(f).convert("RGB").resize((IMG, IMG))
        X.append(np.asarray(img, dtype=np.uint8))
        y.append(indice)
X, y = np.stack(X), np.array(y)
print(f"Caricate {len(X)} immagini {X.shape[1:]} in {time.time() - t0:.0f} s")

# Divisione train / validation / test (stratificata: stesse proporzioni di classi ovunque).
# Validation e test hanno dimensioni multiple di BATCH: sulla TPU i batch tutti uguali
# evitano di ricompilare il modello (vedi più sotto).
n_valtest = (int(0.15 * len(X)) // BATCH) * BATCH
x_train, x_tmp, y_train, y_tmp = train_test_split(X, y, test_size=2 * n_valtest, stratify=y, random_state=42)
x_val, x_test, y_val, y_test = train_test_split(x_tmp, y_tmp, test_size=n_valtest, stratify=y_tmp, random_state=42)
print(f"train: {len(x_train)} | validation: {len(x_val)} | test: {len(x_test)}")

In [ ]:
# Diamo un'occhiata ai dati
fig, assi = plt.subplots(2, 6, figsize=(14, 5))
for ax, i in zip(assi.flat, np.random.default_rng(0).choice(len(x_train), 12, replace=False)):
    ax.imshow(x_train[i])
    ax.set_title(CLASSI[y_train[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()

conteggi = pd.Series([CLASSI[c] for c in y_train]).value_counts()
print("Immagini per classe (train):")
print(conteggi.to_string())

---
## 3. 🛠️ Funzioni di supporto

Scriviamo una sola funzione per allenare **tutti** i modelli nello stesso modo, così il confronto è equo. La funzione salva in una tabella:
- l'accuratezza sul **test set** (immagini mai viste durante l'allenamento)
- il tempo di allenamento
- il numero di **parametri allenabili**

> 💡 **Nota sulla TPU:** al primo passo di ogni modello JAX **compila** il calcolo per la TPU (XLA). La prima epoca è quindi più lenta, mentre le successive sono molto veloci. Il tempo che misuriamo include anche la compilazione.

Usiamo anche la **data augmentation**: durante l'allenamento ogni immagine viene leggermente ruotata, specchiata o zoomata a caso. In questo modo il modello vede "nuove" immagini a ogni epoca e va meno in overfitting.

In [ ]:
RISULTATI = []    # qui raccogliamo i risultati di tutti gli esperimenti
STORIE = {}       # curve di apprendimento per i grafici

def crea_augmentation():
    return keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.1),
    ], name="augmentation")

def conta_parametri(pesi):
    return int(sum(np.prod(p.shape) for p in pesi))

def allena_e_valuta(modello, nome, epoche, lr, x_tr=None, y_tr=None, verbose=1):
    x_tr = x_train if x_tr is None else x_tr
    y_tr = y_train if y_tr is None else y_tr
    # Tagliamo il train a un multiplo di BATCH: tutti i batch uguali = nessuna ricompilazione
    if len(x_tr) > BATCH:
        n = (len(x_tr) // BATCH) * BATCH
        x_tr, y_tr = x_tr[:n], y_tr[:n]

    modello.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])
    t0 = time.time()
    storia = modello.fit(x_tr, y_tr, validation_data=(x_val, y_val),
                         epochs=epoche, batch_size=BATCH, verbose=verbose)
    durata = time.time() - t0
    _, acc_test = modello.evaluate(x_test, y_test, batch_size=BATCH, verbose=0)

    RISULTATI.append({
        "esperimento": nome,
        "accuratezza test": round(float(acc_test), 3),
        "tempo (s)": round(durata),
        "epoche": epoche,
        "immagini train": len(x_tr),
        "parametri allenabili": conta_parametri(modello.trainable_weights),
        "parametri totali": conta_parametri(modello.weights),
    })
    STORIE[nome] = storia.history
    print(f"\n👉 {nome}: accuratezza sul test = {acc_test:.1%}  (tempo: {durata:.0f} s)")
    return storia

---
## 4. 🅰️ Esperimento A: CNN da zero (il punto di partenza)

Una CNN classica come quella della lezione precedente: 4 blocchi Convoluzione + MaxPooling e un classificatore finale. Tutti i pesi partono **casuali**.

È il nostro termine di paragone.

In [ ]:
def crea_cnn_da_zero():
    inputs = keras.Input(shape=(IMG, IMG, 3))
    x = crea_augmentation()(inputs)
    x = layers.Rescaling(1 / 255)(x)                  # pixel da [0, 255] a [0, 1]
    for filtri in [32, 64, 128, 256]:
        x = layers.Conv2D(filtri, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(N_CLASSI, activation="softmax", dtype="float32")(x)
    return keras.Model(inputs, outputs, name="cnn_da_zero")

EPOCHE_DA_ZERO = 3 if MODALITA_VELOCE else 15

cnn = crea_cnn_da_zero()
cnn.summary()
allena_e_valuta(cnn, "A - CNN da zero", epoche=EPOCHE_DA_ZERO, lr=1e-3);

---
## 5. 📦 Il modello pre-allenato: MobileNetV2

**MobileNetV2** è una rete di Google pensata per girare anche sugli smartphone. È stata allenata su ImageNet.

Keras la scarica con una riga di codice:
- `weights="imagenet"` → carica i pesi già allenati
- `include_top=False` → **togliamo** il classificatore originale da 1000 classi, perché ne metteremo uno nostro da 5

Altri modelli disponibili si trovano su [keras.io/api/applications](https://keras.io/api/applications/), e moltissimi altri su Hugging Face e Kaggle Models.

In [ ]:
BACKBONE_DISPONIBILI = {
    # nome: (classe Keras, preprocessing richiesto dal modello)
    "mobilenetv2":    (keras.applications.MobileNetV2,    "meno1_piu1"),   # pixel in [-1, 1]
    "resnet50v2":     (keras.applications.ResNet50V2,     "meno1_piu1"),   # pixel in [-1, 1]
    "efficientnetb0": (keras.applications.EfficientNetB0, "nessuno"),      # normalizza da sola [0, 255]
}

def crea_backbone(nome="mobilenetv2"):
    classe, preproc = BACKBONE_DISPONIBILI[nome]
    base = classe(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
    if preproc == "meno1_piu1":
        preprocessing = layers.Rescaling(1 / 127.5, offset=-1)
    else:
        preprocessing = layers.Identity()
    return base, preprocessing

base, _ = crea_backbone("mobilenetv2")
print(f"Strati:     {len(base.layers)}")
print(f"Parametri:  {conta_parametri(base.weights):,}")
print(f"Output della rete (le 'caratteristiche' estratte): {base.output.shape}")

---
## 6. 🅱️ Esperimento B: Feature extraction

Costruiamo il modello completo:

```
immagine → augmentation → preprocessing → MobileNetV2 (CONGELATA ❄️) → media globale → Dropout → Dense(5)
```

Due dettagli importanti:
- `base.trainable = False` → i pesi di MobileNetV2 **non vengono aggiornati**.
- `base(x, training=False)` → la rete gira sempre in "modalità inferenza". Serve per i layer di **BatchNormalization**: se si mettessero a ricalcolare le loro statistiche sui nostri pochi dati, rovinerebbero quello che il modello ha imparato. Questo vale anche più avanti, durante il fine-tuning.

In [ ]:
def crea_modello_pre_allenato(nome_backbone="mobilenetv2", dropout=0.2):
    base, preprocessing = crea_backbone(nome_backbone)
    base.trainable = False                       # ❄️ congeliamo tutta la rete pre-allenata

    inputs = keras.Input(shape=(IMG, IMG, 3))
    x = crea_augmentation()(inputs)
    x = preprocessing(x)
    x = base(x, training=False)                  # BatchNorm sempre in modalità inferenza
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(N_CLASSI, activation="softmax", dtype="float32")(x)
    return keras.Model(inputs, outputs, name=f"transfer_{nome_backbone}"), base

EPOCHE_FE = 2 if MODALITA_VELOCE else 8

modello, base = crea_modello_pre_allenato("mobilenetv2")
print(f"Parametri allenabili: {conta_parametri(modello.trainable_weights):,}  "
      f"su {conta_parametri(modello.weights):,} totali")
allena_e_valuta(modello, "B - Feature extraction", epoche=EPOCHE_FE, lr=1e-3);

👀 **Osservate:**
- Alleniamo solo poche migliaia di parametri, eppure l'accuratezza è molto più alta della CNN da zero.
- Già dopo la **prima epoca** l'accuratezza di validation è alta. Il modello "sa già vedere": deve solo imparare a dare un nome a quello che vede.

---
## 7. 🅲 Esperimento C: Fine-tuning

Partiamo dal modello dell'esperimento B, che ha già un buon classificatore, e:
1. **Scongeliamo** gli ultimi blocchi di MobileNetV2, quelli con le caratteristiche più "specifiche".
2. Lasciamo congelati i primi strati, che riconoscono bordi e colori e vanno bene così.
3. Usiamo un learning rate **100 volte più basso**: `1e-5` invece di `1e-3`.

**Perché prima B e poi C, e non subito il fine-tuning?**
All'inizio il classificatore nuovo ha pesi casuali e produce errori (gradienti) enormi. Se la rete fosse già scongelata, quei gradienti "rovinerebbero" i pesi pre-allenati. Prima si stabilizza il classificatore, poi si rifinisce il resto.

> ⚠️ Dopo aver cambiato `trainable` bisogna **ricompilare** il modello. Lo fa già la nostra funzione `allena_e_valuta`.

In [ ]:
STRATI_DA_SCONGELARE = 40      # su circa 150 strati di MobileNetV2

base.trainable = True
for strato in base.layers[:-STRATI_DA_SCONGELARE]:
    strato.trainable = False                     # i primi strati restano congelati

print(f"Parametri allenabili ora: {conta_parametri(modello.trainable_weights):,}")

EPOCHE_FT = 2 if MODALITA_VELOCE else 8
allena_e_valuta(modello, "C - Fine-tuning (dopo B)", epoche=EPOCHE_FT, lr=1e-5);

---
## 8. 📊 Confronto A vs B vs C

In [ ]:
tabella = pd.DataFrame(RISULTATI).set_index("esperimento")
display(tabella)

fig, assi = plt.subplots(1, 2, figsize=(14, 4.5))
for nome, storia in STORIE.items():
    assi[0].plot(range(1, len(storia["val_accuracy"]) + 1), storia["val_accuracy"], marker="o", label=nome)
assi[0].set(title="Accuratezza di validation per epoca", xlabel="epoca", ylabel="accuratezza", ylim=(0, 1))
assi[0].legend()
assi[0].grid(alpha=0.3)

assi[1].bar(tabella.index, tabella["accuratezza test"], color=["#999999", "#4C72B0", "#2CA02C"][:len(tabella)])
for i, v in enumerate(tabella["accuratezza test"]):
    assi[1].text(i, v + 0.01, f"{v:.1%}", ha="center", fontweight="bold")
assi[1].set(title="Accuratezza sul test set", ylim=(0, 1.05))
plt.tight_layout()
plt.show()

---
## 9. 🅳 Esperimento D: quanti dati servono davvero?

Questo è il vantaggio più importante nel mondo reale: **etichettare i dati costa**, in tempo, soldi ed esperti.

Ripetiamo A (da zero) e B (feature extraction) usando solo il **10%** e il **30%** delle immagini di train, con lo stesso numero di epoche. Il test set resta sempre lo stesso.

In [ ]:
FRAZIONI = [0.1, 0.3]
for frazione in FRAZIONI:
    x_sub, _, y_sub, _ = train_test_split(x_train, y_train, train_size=frazione,
                                          stratify=y_train, random_state=0)
    print(f"\n===== {frazione:.0%} dei dati ({len(x_sub)} immagini, ~{len(x_sub) // N_CLASSI} per classe) =====")
    allena_e_valuta(crea_cnn_da_zero(), f"A - da zero ({frazione:.0%} dati)",
                    epoche=EPOCHE_DA_ZERO, lr=1e-3, x_tr=x_sub, y_tr=y_sub, verbose=0)
    m, _ = crea_modello_pre_allenato("mobilenetv2")
    allena_e_valuta(m, f"B - feature extraction ({frazione:.0%} dati)",
                    epoche=EPOCHE_FE, lr=1e-3, x_tr=x_sub, y_tr=y_sub, verbose=0)

In [ ]:
def accuratezza(nome):
    return next(r["accuratezza test"] for r in RISULTATI if r["esperimento"] == nome)

percentuali = [f * 100 for f in FRAZIONI] + [100]
acc_zero = [accuratezza(f"A - da zero ({f:.0%} dati)") for f in FRAZIONI] + [accuratezza("A - CNN da zero")]
acc_fe   = [accuratezza(f"B - feature extraction ({f:.0%} dati)") for f in FRAZIONI] + [accuratezza("B - Feature extraction")]

plt.figure(figsize=(8, 5))
plt.plot(percentuali, acc_zero, "o-", color="#999999", lw=2, label="A - CNN da zero")
plt.plot(percentuali, acc_fe, "o-", color="#4C72B0", lw=2, label="B - MobileNetV2 pre-allenata")
for xs, ys in [(percentuali, acc_zero), (percentuali, acc_fe)]:
    for a, b in zip(xs, ys):
        plt.annotate(f"{b:.0%}", (a, b), textcoords="offset points", xytext=(0, 8), ha="center")
plt.xlabel("% delle immagini di train usate")
plt.ylabel("accuratezza sul test")
plt.title("Quanti dati servono?")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.show()

---
## 10. 🔍 Dove sbaglia il modello migliore?

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

probabilita = modello.predict(x_test, batch_size=BATCH, verbose=0)
y_pred = probabilita.argmax(axis=1)

# Matrice di confusione
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=CLASSI,
                                        cmap="Blues", ax=ax, xticks_rotation=30, colorbar=False)
ax.set_title("Matrice di confusione (test set) - modello C")
plt.show()

# Alcuni errori: quali fiori confonde?
errori = np.where(y_pred != y_test)[0][:12]
if len(errori):
    fig, assi = plt.subplots(2, 6, figsize=(15, 5.5))
    for ax in assi.flat:
        ax.axis("off")
    for ax, i in zip(assi.flat, errori):
        ax.imshow(x_test[i])
        ax.set_title(f"vero: {CLASSI[y_test[i]]}\npredetto: {CLASSI[y_pred[i]]}\n({probabilita[i].max():.0%})",
                     color="red", fontsize=9)
    plt.suptitle("Esempi di errori", fontweight="bold")
    plt.tight_layout()
    plt.show()

Spesso gli errori sono "comprensibili": foto con più fiori, fiori lontani, rose e tulipani dello stesso colore, oppure etichette sbagliate nel dataset stesso.

---
## 11. 📝 Riassunto: quando usare cosa

| | Da zero | Feature extraction | Fine-tuning |
|---|---|---|---|
| **Dati necessari** | tanti | pochi (anche decine per classe) | pochi o medi |
| **Tempo di allenamento** | lungo | brevissimo | medio |
| **Rischio di overfitting** | alto | basso | medio (serve un learning rate basso) |
| **Accuratezza tipica** | la più bassa con pochi dati | buona | la migliore |
| **Quando** | dominio molto diverso e tantissimi dati | primo tentativo, sempre | quando la feature extraction non basta |

**✅ Vantaggi del transfer learning**
- Servono molti meno dati etichettati.
- Si allena in minuti invece che in giorni, anche su hardware gratuito.
- I risultati sono migliori e più stabili.
- È lo standard nell'industria: si fa così per immagini, testo (BERT, LLM) e audio.

**⚠️ Limiti e attenzioni**
- **Dominio diverso:** se le vostre immagini sono molto diverse da ImageNet (radiografie, immagini satellitari, termocamere), il vantaggio si riduce. In quei casi il fine-tuning diventa più importante.
- **Il modello è grande:** MobileNetV2 è piccola, ma altri modelli pre-allenati pesano centinaia di MB, e questo conta per il deploy.
- **Il formato dell'input è imposto:** dimensioni e preprocessing devono essere quelli attesi dal modello.
- **Licenze:** controllate sempre la licenza del modello e dei dati su cui è stato allenato, prima di usarlo in un prodotto.
- **Learning rate troppo alto nel fine-tuning:** la rete "dimentica" quello che sapeva (*catastrophic forgetting*). Lo provate nella sfida!

---
## 12. Esercizio

 Modificate i parametri nella cella qui sotto per ottenere la **migliore accuratezza sul test set**.

**Cosa potete cambiare:**
| Parametro | Idee |
|---|---|
| `BACKBONE` | `"mobilenetv2"`, `"efficientnetb0"`, `"resnet50v2"` (più grande: più lenta, ma più potente?) |
| `EPOCHE_FE`, `EPOCHE_FT` | più epoche = meglio? Guardate se la validation smette di salire |
| `LR_FE`, `LR_FT` | provate `LR_FT = 1e-3`: cosa succede? E `1e-4`? |
| `STRATI_DA_SCONGELARE` | 0 = solo feature extraction, `None` = tutta la rete |
| `DROPOUT` | tra 0 e 0.5 |

**Regole:**
1. Non toccate il test set durante le scelte: per decidere usate la **validation**.
2. Annotate ogni tentativo: cosa avete cambiato e che risultato avete ottenuto.
3. Alla fine scrivete il vostro miglior risultato sul test, e la configurazione usata, nella chat del corso.

**Domande a cui rispondere:**
- Il modello più grande è sempre il migliore? E quanto costa in tempo?
- Con `LR_FT = 1e-3` il fine-tuning migliora o peggiora rispetto alla sola feature extraction? Perché?
- Conviene scongelare tutta la rete o solo una parte?

In [ ]:
# ================== 🎛️ I VOSTRI PARAMETRI ==================
BACKBONE = "mobilenetv2"        # "mobilenetv2" | "efficientnetb0" | "resnet50v2"
DROPOUT = 0.2
EPOCHE_FE, LR_FE = 5, 1e-3      # fase 1: feature extraction
EPOCHE_FT, LR_FT = 5, 1e-5      # fase 2: fine-tuning
STRATI_DA_SCONGELARE = 40       # 0 = niente fine-tuning, None = tutta la rete
# ===========================================================

mio_modello, mia_base = crea_modello_pre_allenato(BACKBONE, dropout=DROPOUT)
print(f"Backbone {BACKBONE}: {len(mia_base.layers)} strati, {conta_parametri(mia_base.weights):,} parametri")

inizio = len(RISULTATI)

# Fase 1: feature extraction
allena_e_valuta(mio_modello, f"SFIDA {BACKBONE} - fase 1", epoche=EPOCHE_FE, lr=LR_FE, verbose=2)

# Fase 2: fine-tuning
if STRATI_DA_SCONGELARE != 0 and EPOCHE_FT > 0:
    mia_base.trainable = True
    if STRATI_DA_SCONGELARE is not None:
        for strato in mia_base.layers[:-STRATI_DA_SCONGELARE]:
            strato.trainable = False
    allena_e_valuta(mio_modello, f"SFIDA {BACKBONE} - fase 2", epoche=EPOCHE_FT, lr=LR_FT, verbose=2)

finale = RISULTATI[-1]
print("\n" + "=" * 55)
print(f"🏁 RISULTATO FINALE: {finale['accuratezza test']:.1%} sul test set")
print(f"   tempo totale: {sum(r['tempo (s)'] for r in RISULTATI[inizio:])} s")
print("=" * 55)

In [ ]:
# Tutti i tentativi fatti finora (compresi quelli della sfida)
display(pd.DataFrame(RISULTATI).set_index("esperimento").sort_values("accuratezza test", ascending=False))

---


1. **Testate il modello su una vostra foto.** Caricatela dal pannello *File* a sinistra e usate:
   ```python
   img = np.asarray(Image.open("mia_foto.jpg").convert("RGB").resize((IMG, IMG)))
   p = mio_modello.predict(img[None], verbose=0)[0]
   for c, v in sorted(zip(CLASSI, p), key=lambda t: -t[1]):
       print(f"{c:12s} {v:.1%}")
   ```
   Cosa succede se caricate una foto che **non** è un fiore? Il modello lo capisce?

